In [1]:
import pandas as pd
import re

In [291]:
df = pd.read_csv(r"PGIM-Graph\dataset\URA_merged_full_v1.csv")
print(df['Project Name'].nunique())
print(len(df.columns.tolist()))
for col in df.columns:
    print(col)

3239
367
Project Name
onemap_address
latitude
longitude
Street Name
Postal District
Property Type
No of Bedroom
Monthly Rent ($)
Floor Area (SQM)
Floor Area (SQFT)
Lease Commencement Date
district
Nanyang Primary School
Rosyth School
Henry Park Primary School
Tao Nan School
Raffles Girls' Primary School
St. Hilda's Primary School
Pei Hwa Presbyterian Primary School
Methodist Girls' School (Primary)
Nan Hua Primary School
Chij St. Nicholas Girls' School
Anglo-Chinese School (Primary)
Catholic High School (Primary)
Rulanh Primary School
Red Swastika School
Ai Tong School
St. Joseph's Institution Junior
Kong Hwa School
South View Primary School
Chongfu School
Pei Chun Public School
Holy Innocents' Primary School
Maris Stella High School (Primary)
Singapore Chinese Girls' Primary School
Canberra Primary School
Radin Mas Primary School
River Valley Primary School
Gongshang Primary School
Temasek Primary School
Anderson Primary School
Princess Elizabeth Primary School
Admiralty_MRT
Aljunied_

In [292]:
# df.head()

## Central Core Region

In [293]:
dff = df[df["Postal District"].isin([1, 2, 9, 10, 11])]
dff.tail()

,Project Name,onemap_address,latitude,longitude,Street Name,Postal District,Property Type,No of Bedroom,Monthly Rent ($),Floor Area (SQM),...,1st Post Offices dist,2nd Post Offices,2nd Post Offices dist,3rd Post Offices,3rd Post Offices dist,4th Post Offices,4th Post Offices dist,5th Post Offices,5th Post Offices dist,Project Name in Realis
1409578,VIVA,2 SUFFOLK WALK VIVA SINGAPORE 307462,1.315656,103.844445,SUFFOLD WALK,11,Non-landed Properties,3.0,9200,140 to 150,...,438 m,Novena Post Office,884 m,Post Box,535 m,Newton Post Office,583 m,SingPost Post Box,1.1 km,VIVA
1409579,VIVA,2 SUFFOLK WALK VIVA SINGAPORE 307462,1.315656,103.844445,SUFFOLD WALK,11,Non-landed Properties,3.0,8300,140 to 150,...,438 m,Novena Post Office,884 m,Post Box,535 m,Newton Post Office,583 m,SingPost Post Box,1.1 km,VIVA
1409580,WATTEN HILL,10 WATTEN VIEW WATTEN HILL SINGAPORE 287124,1.330113,103.809575,WATTEN VIEW,11,Non-landed Properties,2.0,6900,240 to 250,...,1.7 km,Lily Avenue postbox,2.1 km,Oro page Show,7.6 km,SingPost Post Box at Holland Ave (posting only),4.3 km,Hellenic Register Asia,3.5 km,WATTEN HILL
1409581,WATTEN RESIDENCES,20 WATTEN RISE WATTEN RESIDENCES SINGAPORE 287302,1.328520,103.809123,WATTEN RISE,11,Semi-Detached House,NaN,11000,400 to 450,...,1.4 km,Lily Avenue postbox,1.8 km,SingPost Post Box at Holland Ave (posting only),4.1 km,SingPost Post Box at Holland Rd Caltex petrol ...,3.8 km,Oro page Show,8.2 km,WATTEN RESIDENCES
1409582,ZEDGE,2 AKYAB ROAD ZEDGE SINGAPORE 309973,1.322422,103.848172,AKYAB ROAD,11,Non-landed Properties,2.0,3700,60 to 70,...,380 m,SingPost Post Box,1.3 km,Novena Post Office,1.7 km,Singapore Post - Novena Branch,1.6 km,Singapore Post - Whampoa Branch,1.7 km,ZEDGE


In [294]:
dff["Lease Commencement Date"] = pd.to_datetime(dff["Lease Commencement Date"], errors="coerce")
dff["year"] = dff["Lease Commencement Date"].dt.year
year_counts = dff["year"].value_counts().sort_index()
print(year_counts)

year
1999.0       47
2000.0     5667
2001.0     6605
2002.0     6676
2003.0     7304
2004.0     8069
2005.0     8352
2006.0     6944
2007.0     6429
2008.0     9612
2009.0    11347
2010.0    12135
2011.0    13527
2012.0    14656
2013.0    16002
2014.0    17077
2015.0    19690
2016.0    19118
2017.0    20969
2018.0    22325
2019.0    23653
2020.0    22182
2021.0    25473
2022.0    22676
2023.0    20555
2024.0    21368
2025.0    22412
2026.0     2061
Name: count, dtype: int64


C:\Users\cecel\AppData\Local\Temp\ipykernel_11760\2832372109.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["year"] = dff["Lease Commencement Date"].dt.year


### Rent per sqft

In [295]:
def sqft_median(x):
    if pd.isna(x):
        return None
    
    x = str(x).replace(",", "")  # remove commas
    
    if "to" in x:
        low, high = x.split("to")
        return (float(low) + float(high)) / 2
    else:
        return float(x)  # if single value
dff["FloorArea_median_sqft"] = dff["Floor Area (SQFT)"].apply(sqft_median)
dff["rent_per_sqft"] = dff["Monthly Rent ($)"] / dff["FloorArea_median_sqft"]

C:\Users\cecel\AppData\Local\Temp\ipykernel_11760\217578015.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["FloorArea_median_sqft"] = dff["Floor Area (SQFT)"].apply(sqft_median)
C:\Users\cecel\AppData\Local\Temp\ipykernel_11760\217578015.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["rent_per_sqft"] = dff["Monthly Rent ($)"] / dff["FloorArea_median_sqft"]


In [296]:
cols_to_drop = ["onemap_address", "Street Name", "Project Name in Realis", "district", "Floor Area (SQM)", "Monthly Rent ($)", "Floor Area (SQFT)", 
                "FloorArea_median_sqft", "Neighbourhood",
    "1st Bus Stops",    "2nd Bus Stops",    "3rd Bus Stops",    "4th Bus Stops",    "5th Bus Stops",
    "1st Supermarkets",    "2nd Supermarkets",    "3rd Supermarkets",    "4th Supermarkets",    "5th Supermarkets",
    "1st Parks",    "2nd Parks",    "3rd Parks",    "4th Parks",    "5th Parks",
    "1st Clinics",    "2nd Clinics",    "3rd Clinics",    "4th Clinics",    "5th Clinics",
    "1st Bank",    "2nd Bank",    "3rd Bank",    "4th Bank",    "5th Bank",
    "1st ATMs",    "2nd ATMs",    "3rd ATMs",    "4th ATMs",    "5th ATMs",
    "1st Post Boxes",    "2nd Post Boxes",    "3rd Post Boxes",    "4th Post Boxes",    "5th Post Boxes",
    "1st Post Offices",    "2nd Post Offices",    "3rd Post Offices",    "4th Post Offices",    "5th Post Offices",
]
dff = dff.drop(columns=cols_to_drop, errors="ignore")

categorical_cols = ["Postal District", "Property Type", 'Planning Area', 'Planning Region']
dff = pd.get_dummies(
    dff,
    columns=categorical_cols,
    prefix=categorical_cols,
    dummy_na=False,
    dtype=int
)

dff.tail()

,Project Name,latitude,longitude,No of Bedroom,Lease Commencement Date,Nanyang Primary School,Rosyth School,Henry Park Primary School,Tao Nan School,Raffles Girls' Primary School,...,Planning Area_Orchard,Planning Area_Outram,Planning Area_Queenstown,Planning Area_River Valley,Planning Area_Rochor,Planning Area_Singapore River,Planning Area_Tanglin,Planning Area_Toa Payoh,Planning Region_Central Region,Planning Region_North East Region
1409578,VIVA,1.315656,103.844445,3.0,2026-01-01,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,0
1409579,VIVA,1.315656,103.844445,3.0,2026-01-01,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,0
1409580,WATTEN HILL,1.330113,103.809575,2.0,2026-01-01,1.062493,NaN,NaN,NaN,0.352918,...,0,0,0,0,0,0,0,0,1,0
1409581,WATTEN RESIDENCES,1.328520,103.809123,NaN,2026-01-01,0.878817,NaN,NaN,NaN,0.349976,...,0,0,0,0,0,0,0,0,1,0
1409582,ZEDGE,1.322422,103.848172,2.0,2026-01-01,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,1,0


### Distance processing

In [297]:
dist_cols = [col for col in dff.columns if " dist" in col]
print(dist_cols)
mrt_cols = [col for col in dff.columns if "_MRT" in col]
print(mrt_cols)
scl_cols = [col for col in dff.columns if "School" in col or "Institution" in col]
print(len(scl_cols))

['1st Bus Stops dist', '2nd Bus Stops dist', '3rd Bus Stops dist', '4th Bus Stops dist', '5th Bus Stops dist', '1st Supermarkets dist', '2nd Supermarkets dist', '3rd Supermarkets dist', '4th Supermarkets dist', '5th Supermarkets dist', '1st Parks dist', '2nd Parks dist', '3rd Parks dist', '4th Parks dist', '5th Parks dist', '1st Clinics dist', '2nd Clinics dist', '3rd Clinics dist', '4th Clinics dist', '5th Clinics dist', '1st Bank dist', '2nd Bank dist', '3rd Bank dist', '4th Bank dist', '5th Bank dist', '1st ATMs dist', '2nd ATMs dist', '3rd ATMs dist', '4th ATMs dist', '5th ATMs dist', '1st Post Boxes dist', '2nd Post Boxes dist', '3rd Post Boxes dist', '4th Post Boxes dist', '5th Post Boxes dist', '1st Post Offices dist', '2nd Post Offices dist', '3rd Post Offices dist', '4th Post Offices dist', '5th Post Offices dist']
['Admiralty_MRT', 'Aljunied_MRT', 'Ang Mo Kio_MRT', 'Bangkit_MRT', 'Bartley_MRT', 'Bayfront_MRT', 'Bayshore_MRT', 'Beauty World_MRT', 'Bedok_MRT', 'Bedok North_MRT'

In [298]:
def convert_to_meters(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()
    
    # extract numeric value
    match = re.search(r"[\d\.]+", x)
    if not match:
        return None
    
    value = float(match.group())
    
    # convert based on unit
    if "km" in x:
        return value
    elif "m" in x:
        return value / 1000
    else:
        return None  # unknown unit
        
for col in dist_cols:
    dff[col] = dff[col].apply(convert_to_meters)
dff[mrt_cols] = dff[mrt_cols] / 1000

dff[dist_cols] = dff[dist_cols].fillna(5)
dff[mrt_cols] = dff[mrt_cols].fillna(5)
dff[scl_cols] = dff[scl_cols].fillna(3)

dff.tail()

,Project Name,latitude,longitude,No of Bedroom,Lease Commencement Date,Nanyang Primary School,Rosyth School,Henry Park Primary School,Tao Nan School,Raffles Girls' Primary School,...,Planning Area_Orchard,Planning Area_Outram,Planning Area_Queenstown,Planning Area_River Valley,Planning Area_Rochor,Planning Area_Singapore River,Planning Area_Tanglin,Planning Area_Toa Payoh,Planning Region_Central Region,Planning Region_North East Region
1409578,VIVA,1.315656,103.844445,3.0,2026-01-01,3.000000,3.0,3.0,3.0,3.000000,...,0,0,0,0,0,0,0,0,1,0
1409579,VIVA,1.315656,103.844445,3.0,2026-01-01,3.000000,3.0,3.0,3.0,3.000000,...,0,0,0,0,0,0,0,0,1,0
1409580,WATTEN HILL,1.330113,103.809575,2.0,2026-01-01,1.062493,3.0,3.0,3.0,0.352918,...,0,0,0,0,0,0,0,0,1,0
1409581,WATTEN RESIDENCES,1.328520,103.809123,NaN,2026-01-01,0.878817,3.0,3.0,3.0,0.349976,...,0,0,0,0,0,0,0,0,1,0
1409582,ZEDGE,1.322422,103.848172,2.0,2026-01-01,3.000000,3.0,3.0,3.0,3.000000,...,0,0,0,0,0,0,0,0,1,0


### Missing value processing

In [299]:
has_cols = [col for col in dff.columns if "Has_" in col]
print(len(has_cols))
dff[has_cols] = dff[has_cols].fillna(0)

64


In [300]:
dff = dff.dropna(subset=["Lease Commencement Date"])
for col in ["Condo_Age_2026", "house_age_2026", "tenure_remaining_years"]:
    dff[col] = dff[col].fillna(dff[col].median())
dff["No of Bedroom_missing"] = dff["No of Bedroom"].isna().astype(int)
dff["Large_Dev_missing"] = dff["Large_Dev_200plus"].isna().astype(int)
fill_neg_cols = ["No of Bedroom"]
dff[fill_neg_cols] = dff[fill_neg_cols].fillna(-1)
fill_zero_cols = ["project_size_small", "project_size_medium", "project_size_large", "Large_Dev_200plus"]
dff[fill_zero_cols] = dff[fill_zero_cols].fillna(0)

cols = ["sora_overnight_pct", "sora_compounded_3m_pct", "ura_private_rental_index_yoy_pct", "ura_private_price_index_avg_3seg_yoy_pct"]
dff = dff.sort_values("Lease Commencement Date")
dff[cols] = dff[cols].ffill().bfill()


C:\Users\cecel\AppData\Local\Temp\ipykernel_11760\2840926910.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["No of Bedroom_missing"] = dff["No of Bedroom"].isna().astype(int)
C:\Users\cecel\AppData\Local\Temp\ipykernel_11760\2840926910.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dff["Large_Dev_missing"] = dff["Large_Dev_200plus"].isna().astype(int)


In [301]:
dff

,Project Name,latitude,longitude,No of Bedroom,Lease Commencement Date,Nanyang Primary School,Rosyth School,Henry Park Primary School,Tao Nan School,Raffles Girls' Primary School,...,Planning Area_Queenstown,Planning Area_River Valley,Planning Area_Rochor,Planning Area_Singapore River,Planning Area_Tanglin,Planning Area_Toa Payoh,Planning Region_Central Region,Planning Region_North East Region,No of Bedroom_missing,Large_Dev_missing
114592,FAR EAST PLAZA,1.307177,103.833793,-1.0,1999-11-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,0,0,0,0,0,0,1,1
205397,OEI TIONG HAM PARK,1.315055,103.791001,-1.0,1999-12-01,1.839042,3.0,0.748327,3.0,3.0,...,0,0,0,0,0,0,1,0,1,1
244,EMERALD GARDEN,1.281549,103.846473,-1.0,1999-12-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,0,0,0,0,1,0,1,0
124204,MIRAGE TOWER,1.291981,103.834262,-1.0,1999-12-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,0,1,0,0,1,0,1,0
138671,THE ASCOTT,1.310995,103.838733,-1.0,1999-12-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408201,MOUNT SOPHIA SUITES,1.302039,103.846842,1.0,2026-01-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,1,0,0,0,1,0,0,0
1408200,MOUNT SOPHIA SUITES,1.302039,103.846842,1.0,2026-01-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,1,0,0,0,1,0,0,0
1408199,MIRAGE TOWER,1.291981,103.834262,3.0,2026-01-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,0,1,0,0,1,0,0,0
1408165,MACKENZIE 88,1.306472,103.847683,1.0,2026-01-01,3.000000,3.0,3.000000,3.0,3.0,...,0,0,1,0,0,0,1,0,0,0


In [302]:
count_le_2024 = (dff["year"] <= 2023).sum()
count_gt_2024 = (dff["year"] > 2023).sum()

print("<= 2023:", count_le_2024)
print("> 2023:", count_gt_2024)

<= 2023: 347090
> 2023: 45841


In [313]:
def reorder_col(dff):
    print("Before column reordering: ", len(dff.columns.tolist()))
    # --- Base columns ---
    base_cols = [
        "Project Name",
        "Lease Commencement Date",
        "latitude",
        "longitude",
        "No of Bedroom",
        "No of Bedroom_missing",
    ]
    
    # --- Structural + Macro ---
    struct_macro_cols = [
        col for col in dff.columns
        if any(x in col for x in [
            "Large_Dev", "_Age_", "_age_",
            "tenure", "cpi", "unemployment",
            "sora", "bond", "sgd", "ura",
            "hdb", "gva"
        ])
    ]
    
    # --- One-hot ---
    onehot_cols = [
        col for col in dff.columns
        if any(x in col for x in [
            "Postal District",
            "Property Type",
            "Planning Area",
            "Planning Region",
            "project_size"
        ])
    ]
    
    # --- Schools ---
    school_cols = [
        col for col in dff.columns
        if "School" in col or "Institution" in col
    ]
    
    # --- MRT ---
    mrt_cols = [
        col for col in dff.columns
        if "_MRT" in col
    ]
    
    # --- Distances ---
    dist_cols = [
        col for col in dff.columns
        if "dist" in col.lower()
    ]
    
    # --- Facilities ---
    facility_cols = [
        col for col in dff.columns
        if col.startswith("Has_")
    ]
    
    # --- Engineered ---
    engineered_cols = [
        col for col in dff.columns
        if col in ["year", "rent_per_sqft"] or col.endswith("_missing")
    ]
    
    # --- Combine in order ---
    ordered_cols = (
        base_cols
        + struct_macro_cols
        + onehot_cols
        + school_cols
        + mrt_cols
        + dist_cols
        + facility_cols
        + engineered_cols
    )
    
    # --- Remove duplicates while preserving order ---
    ordered_cols = list(dict.fromkeys(ordered_cols))
    
    # --- Add any remaining columns (safety) ---
    remaining_cols = [col for col in dff.columns if col not in ordered_cols]
    ordered_cols += remaining_cols
    
    # --- Reorder dataframe ---
    dff = dff[ordered_cols]
    print("After column reordering: ", len(dff.columns.tolist()))
    return dff
dff = reorder_col(dff)

Before column reordering:  346
After column reordering:  346


In [314]:
# dff.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v1.csv")
dff.describe().to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v1_describe.csv")
dff.describe()

,latitude,longitude,No of Bedroom,No of Bedroom_missing,Lease Commencement Date,Large_Dev_200plus,Condo_Age_2026,tenure_remaining_years,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Underwater Fitness Station,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft
count,392931.000000,392931.000000,392931.000000,392931.000000,392931,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000,...,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000,392931.000000
mean,1.303849,103.830307,1.174914,0.352785,2016-02-12 14:21:45.957534,0.356640,30.104451,7534.085356,0.716747,0.006630,...,0.014908,0.020497,0.431572,0.001771,0.129353,0.025844,0.003649,0.050973,2015.678486,4.394572
min,1.272156,103.721044,-1.000000,0.000000,1999-11-01 00:00:00,0.000000,7.000000,43.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.448966
25%,1.294259,103.825520,-1.000000,0.000000,2011-08-01 00:00:00,0.000000,29.000000,9999.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2011.000000,3.200000
50%,1.303695,103.836025,1.000000,0.000000,2017-05-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2017.000000,4.210526
75%,1.315448,103.841806,3.000000,1.000000,2021-08-01 00:00:00,1.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2021.000000,5.391304
max,1.402715,103.944841,8.000000,1.000000,2026-01-01 00:00:00,1.000000,58.000000,9999.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154
std,0.014824,0.017585,1.795824,0.477837,NaN,0.479008,6.277050,4272.501131,0.450579,0.081152,...,0.121187,0.141694,0.495296,0.042050,0.335591,0.158671,0.060301,0.219944,6.726900,1.669546


In [315]:
dfg = (
    dff.groupby(["Project Name", "Lease Commencement Date"])
      .mean(numeric_only=True)
      .reset_index()
)
dfg = reorder_col(dfg)
# dfg.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v2.csv")
dfg.describe().to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v2_describe.csv")
dfg.describe()

C:\Users\cecel\AppData\Local\Temp\ipykernel_11760\964218318.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


Before column reordering:  346
After column reordering:  346


,latitude,longitude,No of Bedroom,No of Bedroom_missing,Lease Commencement Date,Large_Dev_200plus,Condo_Age_2026,tenure_remaining_years,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Underwater Fitness Station,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft
count,110274.000000,110274.000000,110274.000000,110274.000000,110274,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000,...,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000,110274.000000
mean,1.308660,103.827392,1.011918,0.433971,2015-05-01 08:37:53.725447,0.118106,30.420045,8757.034795,0.809865,0.006076,...,0.006429,0.011408,0.332345,0.002893,0.086131,0.017765,0.003292,0.028230,2014.890817,3.934401
min,1.272156,103.721044,-1.000000,0.000000,1999-11-01 00:00:00,0.000000,7.000000,43.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.461538
25%,1.298505,103.820856,-1.000000,0.000000,2010-03-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2010.000000,2.869565
50%,1.309875,103.832190,1.333333,0.000000,2016-05-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2016.000000,3.753814
75%,1.317478,103.839800,3.000000,1.000000,2021-04-01 00:00:00,0.000000,30.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2021.000000,4.821816
max,1.402715,103.944841,8.000000,1.000000,2026-01-01 00:00:00,1.000000,58.000000,9999.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154
std,0.013001,0.017751,1.859909,0.490710,NaN,0.322735,7.159498,3274.144427,0.392410,0.077710,...,0.079926,0.106197,0.471056,0.053707,0.280559,0.132096,0.057280,0.165629,7.032671,1.530480


In [316]:
dfg

,Project Name,latitude,longitude,No of Bedroom,No of Bedroom_missing,Lease Commencement Date,Large_Dev_200plus,Condo_Age_2026,tenure_remaining_years,tenure_freehold_like,...,Has_Underwater Fitness Station,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft
0,1 MOULMEIN RISE,1.319086,103.847054,3.000000,0.0,2012-01-01,0.0,23.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2012.0,4.695652
1,1 MOULMEIN RISE,1.319086,103.847054,3.000000,0.0,2012-02-01,0.0,23.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2012.0,4.086957
2,1 MOULMEIN RISE,1.319086,103.847054,3.000000,0.0,2012-03-01,0.0,23.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2012.0,4.608696
3,1 MOULMEIN RISE,1.319086,103.847054,3.000000,0.0,2012-06-01,0.0,23.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2012.0,4.478261
4,1 MOULMEIN RISE,1.319086,103.847054,3.000000,0.0,2012-10-01,0.0,23.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2012.0,4.695652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110269,ZENITH,1.292637,103.830568,1.000000,0.0,2025-08-01,0.0,29.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6.573232
110270,ZENITH,1.292637,103.830568,1.333333,0.0,2025-10-01,0.0,29.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6.563636
110271,ZENITH,1.292637,103.830568,2.000000,0.0,2025-11-01,0.0,29.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,4.782609
110272,ZENITH,1.292637,103.830568,1.500000,0.0,2025-12-01,0.0,29.0,9999.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6.252525


In [317]:
count_le_2024 = (dfg["year"] <= 2023).sum()
count_gt_2024 = (dfg["year"] > 2023).sum()

print("<= 2023:", count_le_2024)
print("> 2023:", count_gt_2024)

<= 2023: 98327
> 2023: 11947


In [318]:
# cols = [col for col in df.columns if col != "Lease Commencement Date"]

# constant_by_date_cols = []

# for col in cols:
#     nunique_per_date = df.groupby("Lease Commencement Date")[col].nunique(dropna=False)
    
#     if (nunique_per_date <= 1).all():
#         constant_by_date_cols.append(col)

# print("Columns constant within each date:")
# print(constant_by_date_cols)

In [319]:
nunique_per_date

Lease Commencement Date
1999-11-01       5
1999-12-01      80
2000-01-01     394
2000-02-01     379
2000-03-01     399
              ... 
2025-08-01    1659
2025-10-01    1533
2025-11-01    1485
2025-12-01    1601
2026-01-01    1579
Name: Project Name in Realis, Length: 289, dtype: int64

In [320]:
import pandas as pd

constant_by_date_cols = [
    "cpi_all_items_infl_yoy_pct",
    "unemployment_rate_sa_pct",
    "sora_overnight_pct",
    "sora_compounded_3m_pct",
    "sg_govt_bond_yield_10y_pct",
    "sgd_per_usd_logret_12m_pct",
    "ura_private_rental_index_yoy_pct",
    "ura_private_price_index_avg_3seg_yoy_pct",
    "hdb_resale_price_index_yoy_pct",
    "gva_yoy_growth_pct",
]

target_col = "rent_per_sqft"
group_cols = ["Project Name", "Lease Commencement Date"]

dfg["Lease Commencement Date"] = pd.to_datetime(
    dfg["Lease Commencement Date"], errors="coerce"
)
dfg = dfg.sort_values(["Project Name", "Lease Commencement Date"]).copy()

project_level_cols = [
    col for col in dfg.columns
    if col not in group_cols + constant_by_date_cols + [target_col]
]

# build date-level macro table
macro_by_date = (
    dfg.groupby("Lease Commencement Date")[constant_by_date_cols]
       .first()
       .sort_index()
)

# fill missing months in macro table too
full_macro_dates = pd.date_range(
    start=dfg["Lease Commencement Date"].min(),
    end=dfg["Lease Commencement Date"].max(),
    freq="MS"
)

macro_by_date = macro_by_date.reindex(full_macro_dates)
macro_by_date.index.name = "Lease Commencement Date"
macro_by_date = macro_by_date.interpolate(method="time", limit_direction="both")


def expand_project(project_name, group):
    group = group.sort_values("Lease Commencement Date").copy()

    full_dates = pd.date_range(
        start=group["Lease Commencement Date"].min(),
        end=group["Lease Commencement Date"].max(),
        freq="MS"
    )

    g = (
        group.set_index("Lease Commencement Date")
             .reindex(full_dates)
             .rename_axis("Lease Commencement Date")
             .reset_index()
    )

    g["Project Name"] = project_name

    # ✅ STEP 1: mark original vs missing BEFORE interpolation
    g["y_mask"] = g["rent_per_sqft"].notna().astype(int)

    # fill project-level columns
    existing_project_cols = [c for c in project_level_cols if c in g.columns]
    g[existing_project_cols] = g[existing_project_cols].ffill().bfill()

    # ✅ STEP 2: interpolate rent
    g["rent_per_sqft"] = g["rent_per_sqft"].interpolate(
        method="linear",
        limit_direction="both"
    )

    # merge macro columns
    g = g.drop(columns=constant_by_date_cols, errors="ignore")

    g = g.merge(
        macro_by_date.reset_index(),
        on="Lease Commencement Date",
        how="left"
    )

    return g

expanded_list = []

for project_name, group in dfg.groupby("Project Name"):
    expanded_list.append(expand_project(project_name, group))

dfge = pd.concat(expanded_list, ignore_index=True)
dfge = reorder_col(dfge)
print(dfge.shape)
print(dfge.head())

Before column reordering:  347
After column reordering:  347
(205812, 347)
      Project Name  latitude   longitude  No of Bedroom  \
0  1 MOULMEIN RISE  1.319086  103.847054            3.0   
1  1 MOULMEIN RISE  1.319086  103.847054            3.0   
2  1 MOULMEIN RISE  1.319086  103.847054            3.0   
3  1 MOULMEIN RISE  1.319086  103.847054            3.0   
4  1 MOULMEIN RISE  1.319086  103.847054            3.0   

   No of Bedroom_missing Lease Commencement Date  Large_Dev_200plus  \
0                    0.0              2012-01-01                0.0   
1                    0.0              2012-02-01                0.0   
2                    0.0              2012-03-01                0.0   
3                    0.0              2012-04-01                0.0   
4                    0.0              2012-05-01                0.0   

   Condo_Age_2026  tenure_remaining_years  tenure_freehold_like  ...  \
0            23.0                  9999.0                   1.0  ...   

In [321]:
dfge.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v3.0.csv")
dfge.describe().to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v3.0_describe.csv")
dfge.describe()

,latitude,longitude,No of Bedroom,No of Bedroom_missing,Lease Commencement Date,Large_Dev_200plus,Condo_Age_2026,tenure_remaining_years,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Viewing Deck,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft,y_mask
count,205812.000000,205812.000000,205812.000000,205812.000000,205812,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000,...,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000,205812.000000
mean,1.310449,103.826036,0.695902,0.533009,2014-09-23 19:22:13.076788,0.069724,30.882436,9032.695727,0.790824,0.004363,...,0.007560,0.258440,0.002969,0.061090,0.012186,0.002050,0.019047,2014.063942,3.651005,0.535800
min,1.272156,103.721044,-1.000000,0.000000,1999-11-01 00:00:00,0.000000,7.000000,43.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.461538,0.000000
25%,1.301067,103.813314,-1.000000,0.000000,2009-02-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2008.000000,2.589171,0.000000
50%,1.312907,103.830428,-1.000000,1.000000,2015-07-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2015.000000,3.430303,1.000000
75%,1.318783,103.839257,2.666667,1.000000,2020-11-01 00:00:00,0.000000,32.000000,9999.000000,1.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2020.000000,4.497314,1.000000
max,1.402715,103.944841,8.000000,1.000000,2026-01-01 00:00:00,1.000000,58.000000,9999.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154,1.000000
std,0.012855,0.018236,1.896496,0.495604,NaN,0.254682,7.295776,2932.739738,0.406722,0.065910,...,0.086621,0.437778,0.054405,0.239495,0.109715,0.045235,0.136689,7.201113,1.530934,0.498718


In [322]:
dfge["Lease Commencement Date"] = pd.to_datetime(
    dfge["Lease Commencement Date"], errors="coerce"
)

dfgep = dfge.sort_values(["Project Name", "Lease Commencement Date"]).copy()

# add previous timestep rent_per_sqft
dfgep["prev_y"] = dfgep.groupby("Project Name")["rent_per_sqft"].shift(1)

# remove earliest record of each project (where prev_y is NaN)
dfgep = dfgep[dfgep["prev_y"].notna()].reset_index(drop=True)
dfgep.to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v3.1.csv")
dfgep.describe().to_csv(r"PGIM-Graph\dataset\URA_merged_ccr_v3.1_describe.csv")
dfgep.describe()

,latitude,longitude,No of Bedroom,No of Bedroom_missing,Lease Commencement Date,Large_Dev_200plus,Condo_Age_2026,tenure_remaining_years,tenure_freehold_like,tenure_medium_lease_60_80,...,Has_Wading Pool,Has_Water Channel,Has_Water Feature,Has_Waterfall,Has_Wine And Cigar Room,Has_Yoga Corner,year,rent_per_sqft,y_mask,prev_y
count,204851.000000,204851.000000,204851.000000,204851.000000,204851,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,...,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000,204851.000000
mean,1.310452,103.826030,0.702434,0.531243,2014-10-07 03:31:53.833957,0.069704,30.886171,9032.887411,0.790955,0.004369,...,0.258402,0.002968,0.061015,0.012180,0.002050,0.018955,2014.099072,3.649956,0.533622,3.645505
min,1.272156,103.721044,-1.000000,0.000000,1999-12-01 00:00:00,0.000000,7.000000,43.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1999.000000,0.461538,0.000000,0.461538
25%,1.301067,103.813314,-1.000000,0.000000,2009-03-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2008.000000,2.590476,0.000000,2.586667
50%,1.312907,103.830428,-1.000000,1.000000,2015-08-01 00:00:00,0.000000,29.000000,9999.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2015.000000,3.430454,1.000000,3.428312
75%,1.318783,103.839257,2.666667,1.000000,2020-11-01 00:00:00,0.000000,32.000000,9999.000000,1.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2020.000000,4.494422,1.000000,4.488325
max,1.402715,103.944841,8.000000,1.000000,2026-01-01 00:00:00,1.000000,58.000000,9999.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2026.000000,23.846154,1.000000,23.846154
std,0.012853,0.018234,1.897298,0.495708,NaN,0.254649,7.297005,2932.471296,0.406627,0.065954,...,0.437757,0.054399,0.239359,0.109687,0.045234,0.136367,7.181894,1.528155,0.498869,1.527576


In [9]:
count_le_2024 = (dfgep["year"] <= 2023).sum()
count_gt_2024 = (dfgep["year"] > 2023).sum()

print("<= 2023:", count_le_2024)
print("> 2023:", count_gt_2024)

<= 2023: 186445
> 2023: 18406
